# 01 Load and split (card D1)

## Setup

In [1]:
# Make src/ importable and load project settings and helpers
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import hashlib

import pandas as pd

from stroke_prep import config, pipeline

## Load the raw file

In [2]:
# Show which text tokens stand for missing values in number columns
first_read = pd.read_csv(config.RAW_PATH)
print("missing value tokens found:", pipeline.missing_tokens(first_read))

missing value tokens found: []


In [3]:
# Load the raw CSV with Python, reading those tokens as missing
raw = pipeline.load_raw()
print("shape:", raw.shape)

shape: (5110, 12)


In [4]:
# Column types before any change
raw.dtypes

id                     int64
gender                   str
age                  float64
hypertension           int64
heart_disease          int64
ever_married             str
work_type                str
Residence_type           str
avg_glucose_level    float64
bmi                  float64
smoking_status           str
stroke                 int64
dtype: object

In [5]:
# First rows before any change
raw.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [6]:
# Missing values per column before any change
raw.isna().sum()

id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

In the raw data, bmi is the only column with missing values: 201 of 5,110 rows (about 4%). Every other column shows 0 missing. smoking_status has no blanks, but it may hide missing information as an "Unknown" category, which I check in notebook 02.

## Split 70/30 once (card D1)

In [7]:
# Decision values from config.py (card D1)
print("TEST_SIZE:", config.TEST_SIZE)
print("SPLIT_SEED:", config.SPLIT_SEED)
print("STRATIFY_ON:", config.STRATIFY_ON)

TEST_SIZE: 0.3
SPLIT_SEED: 33
STRATIFY_ON: None


In [8]:
# Split once with the settings above
train, test = pipeline.split_train_test(raw)
print("train rows:", len(train))
print("test rows:", len(test))

train rows: 3577
test rows: 1533


The split gave 3,577 training rows and 1,533 test rows. Together that's 5,110, the full raw file, so no row was lost or counted twice, and the ratio is 70/30 as Canvas asks.

## Shares in train and test (card D1, how to test step 1)

In [9]:
# Share of each value in train and in test, for one column
def shares(col):
    table = pd.DataFrame({
        "train": train[col].value_counts(normalize=True),
        "test": test[col].value_counts(normalize=True),
    })
    return table.round(4)

In [10]:
# stroke shares in train and test
shares(config.TARGET)

,train,test
stroke,,
0,0.953,0.9472
1,0.047,0.0528


In [11]:
# work_type shares in train and test
shares(config.BALANCE_COL)

,train,test
work_type,,
Private,0.5706,0.5766
Self-employed,0.1635,0.1526
children,0.1334,0.1370
Govt_job,0.1289,0.1279
Never_worked,0.0036,0.0059


The shares stayed close between the two parts. Stroke is 4.7% in train and 5.3% in test. work_type shares differ by about one percentage point or less. Never_worked is the smallest group, at 0.36% in train and 0.59% in test, but it's present in both. These small gaps are expected from a plain random split without stratifying.

## Split checks (card D1, how to test steps 2 and 3)

In [12]:
# The parts add up to the raw row count and share no id
assert len(train) + len(test) == len(raw)
assert set(train["id"]).isdisjoint(set(test["id"]))
print("rows add up and no id is in both parts")

rows add up and no id is in both parts


In [13]:
# Every work_type group, including the smallest, appears in both parts
raw_groups = set(raw[config.BALANCE_COL])
assert set(train[config.BALANCE_COL]) == raw_groups
assert set(test[config.BALANCE_COL]) == raw_groups
print("every work_type group is in train and in test")

every work_type group is in train and in test


In [14]:
# Rerun the split with the same seed and confirm the same ids land in train
train_again, _ = pipeline.split_train_test(raw)
assert list(train_again["id"]) == list(train["id"])
print("same seed gives the same train ids")

same seed gives the same train ids


All split checks passed. The parts add up to the raw row count, no id appears in both, every work_type group shows up in train and test, and rerunning with seed 33 puts the same ids in train. This confirms the split is reproducible, as I said in D1.

## Training data in memory

In [15]:
# First rows of the training part
train.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
3339,35182,Female,62.0,0,0,Yes,Govt_job,Rural,98.14,42.0,Unknown,0
2186,20565,Male,13.0,0,0,No,children,Rural,85.87,24.3,Unknown,0
1472,41271,Male,68.0,1,0,Yes,Govt_job,Urban,222.29,30.1,never smoked,0
3178,64597,Female,33.0,0,0,Yes,Private,Rural,73.20,28.9,Unknown,0
4600,50804,Male,2.0,0,0,No,children,Rural,65.84,16.1,Unknown,0


## Save train and test once

In [16]:
# Save both parts; if they were saved before, check they match instead of overwriting
for part, path in [(train, config.TRAIN_RAW), (test, config.TEST_RAW)]:
    if path.exists():
        saved_ids = pd.read_csv(path)["id"].tolist()
        assert saved_ids == part["id"].tolist(), f"{path.name} differs from this split"
        print("already saved, matches:", path.name)
    else:
        pipeline.save_csv(part, path)
        print("saved:", path.name)

already saved, matches: train_raw.csv
already saved, matches: test_raw.csv


## Raw file unchanged

In [17]:
# Compare the raw file's SHA256 with data/raw/SHA256SUMS
expected = (config.RAW_PATH.parent / "SHA256SUMS").read_text().split()[0]
actual = hashlib.sha256(config.RAW_PATH.read_bytes()).hexdigest()
assert actual == expected
print("raw file hash matches SHA256SUMS")

raw file hash matches SHA256SUMS


The raw file's hash matches SHA256SUMS, so the original data was not changed.